***

# **Transit Facilities by Region**

***

This is our version of the DVRPC's Transit Facilities by Region dataset. The dataset details the number of deficient transit fleet conditions by agency, county, and facility type. The data was obtained from the [*Federal Transit Administration's (FTA) National Transit Database (NTD)*](https://www.transit.dot.gov/ntd/ntd-data?field_product_type_target_id=All&year=all&combine=facility). The data encompasses either rail vehicle or bus modes of transportation. Rail vehicles include hybrid rail and trolleys, heavy rail (i.e., subway), and regional rail passenger cars and locomotives.

***

# **Transit Code**

***

In [ ]:
# Packages

import pandas as pd
import os
import itertools

pd.set_option('display.max_columns', None)

In [2]:
# User paths

user = os.getlogin()
path_users = os.path.join('C:\\Users', user)
path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
path_transit = os.path.join(path_sp, 'Products', 'RHNA', 'New Data Collected', 'Transit Conditions')
path_data = os.path.join(path_transit, 'Transit Conditions Data Sources.xlsx')

In [ ]:
# Loading the data
sources = pd.read_excel(path_data)

# Getting the unique IDs NTD that SACOG previously used 
path_ids = os.path.join(path_transit, 'Copy of sacramento_area_council_of_governments.xlsx')
df_ids = pd.read_excel(path_ids)
ids = df_ids['NTD ID'].unique().tolist()

# Filter for 'NTD Facility Inventory'
facility_df = sources[sources["Name"] == "NTD Facility Inventory"]

all_data = []

# Using the sources file, we just loop through the URLs we care about rather than directly having to download the data
for index, row in facility_df.iterrows():
    url = row["URL"]
    year = row["Year"]
    
    try:
        df_excel = pd.read_excel(url)
        df_excel["Year"] = year  # Add the Year column
        all_data.append(df_excel)
        
        print(f"Successfully loaded the data for {year} from {url}")
    except Exception as e:
        print(f"Failed to read {url} for {year}: {e}")

data = pd.concat(all_data, ignore_index=True)

# Filter by agency now, used the same ones as in here: https://ntd-monthly-ridership--cal-itp-data-analyses.netlify.app/rtpa_sacramento-area-council-of-governments/00__monthly_ridership_report__rtpa_sacramento-area-council-of-governments
agency = ['Sacramento Regional Transit District', 'University of California, Davis',
                'Yolo County Transportation District', 'Yuba-Sutter Transit Authority',
                'City of Elk Grove', 'Paratransit, Inc.', 'City of Folsom',
                'County of Sacramento Municipal Services Agency', 'Paratransit, Inc. CTSA'
                ]

data = data[data['NTD ID'].isin(ids)]
# data = data[data['NTD ID'].isin(agency)] also works

data['Facility'] = data.apply(lambda row: 'Admin' if row['Administrative/Maintenance Facility Flag'] == 1 else 'Passenger', axis=1)

# Data Transformation

passenger = data[(data['Facility'] == 'Passenger') & (data['Condition Assessment'] < 3)]
admin = data[(data['Facility'] == 'Admin') & (data['Condition Assessment'] < 3)]

Successfully loaded the data for 2018 from https://www.transit.dot.gov/sites/fta.dot.gov/files/2018%20Facility%20Inventory.xlsx
Successfully loaded the data for 2019 from https://www.transit.dot.gov/sites/fta.dot.gov/files/2020-10/2019%20Facility%20Inventory.xlsx
Successfully loaded the data for 2020 from https://www.transit.dot.gov/sites/fta.dot.gov/files/2021-11/2020%20Facility%20Inventory.xlsx
Successfully loaded the data for 2021 from https://www.transit.dot.gov/sites/fta.dot.gov/files/2022-10/2021%20Facility%20Inventory.xlsx
Successfully loaded the data for 2022 from https://www.transit.dot.gov/sites/fta.dot.gov/files/2023-10/2022%20Facility%20Inventory_0.xlsx
Successfully loaded the data for 2023 from https://www.transit.dot.gov/sites/fta.dot.gov/files/2024-10/2023%20Facility%20Inventory.xlsx


In [ ]:
# Gather the unique values for these for a full combination map
years = data['Year'].unique()
agencies = data['Agency Name'].unique()
facilities = data['Facility'].unique()
ntd_ids = data['NTD ID'].unique()

# This is generating all the combos that can be had given our unique classifiers above
all_combinations = pd.DataFrame(itertools.product(years, agencies, facilities, ntd_ids), 
                                columns=['Year', 'Agency Name', 'Facility', 'NTD ID'])

# Get the total num of facilities
total_facilities = data.groupby(['Year', 'Agency Name', 'Facility', 'NTD ID']).size().reset_index(name='Total Facilities')

# Count deficient facilities (Condition Assessment < 3 as per Ian Schwarzenberg) 
deficient_facilities = data[data['Condition Assessment'] < 3].groupby(['Year', 'Agency Name', 'Facility', 'NTD ID']).size().reset_index(name='Deficient Facilities')

# Merge
facility_summary = all_combinations.merge(total_facilities, on=['Year', 'Agency Name', 'Facility', 'NTD ID'], how='left').fillna(0)
facility_summary = facility_summary.merge(deficient_facilities, on=['Year', 'Agency Name', 'Facility', 'NTD ID'], how='left').fillna(0)

# Calc percents 
facility_summary['Deficient Percentage'] = (facility_summary['Deficient Facilities'] / facility_summary['Total Facilities']).replace({0: 0}) * 100
facility_summary['Deficient Percentage'] = facility_summary['Deficient Percentage'].fillna(0)

display(facility_summary)


,Year,Agency Name,Facility,NTD ID,Total Facilities,Deficient Facilities,Deficient Percentage
0,2018,Sacramento Regional Transit District,Admin,90019,16.0,0.0,0.0
1,2018,Sacramento Regional Transit District,Admin,90061,0.0,0.0,0.0
2,2018,Sacramento Regional Transit District,Admin,90090,0.0,0.0,0.0
3,2018,Sacramento Regional Transit District,Admin,90142,0.0,0.0,0.0
4,2018,Sacramento Regional Transit District,Admin,90205,0.0,0.0,0.0
...,...,...,...,...,...,...,...
583,2023,"Paratransit, Inc.",Passenger,90090,0.0,0.0,0.0
584,2023,"Paratransit, Inc.",Passenger,90142,0.0,0.0,0.0
585,2023,"Paratransit, Inc.",Passenger,90205,0.0,0.0,0.0
586,2023,"Paratransit, Inc.",Passenger,90220,0.0,0.0,0.0


In [19]:
# Export Prep
path_facilities = os.path.join(path_transit, "Transit Facilities by Region")

print(''); print('')
print('Converting/exporting Transit Facilities by Region results to SharePoint')

# Export
filename = f"Transit Facilities by Region"

name_out_xlsx = filename + '.xlsx'
name_out_csv = filename + '.csv'

path_out_xlsx = os.path.join(path_facilities, name_out_xlsx)
path_out_csv = os.path.join(path_facilities, name_out_csv)


with pd.ExcelWriter(os.path.join(path_out_xlsx), engine='xlsxwriter') as writer:
    facility_summary .to_excel(writer, index = False, sheet_name = 'Listings')

facility_summary.to_csv(path_out_csv, index=False)

# print(f"Excel files exported here:  {path_out_xlsx}");print('')
print(f"Files exported here:  {path_facilities}");print('')



Converting/exporting Transit Facilities by Region results to SharePoint
Files exported here:  C:\Users\jchoy\Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents\Products\RHNA\New Data Collected\Transit Conditions\Transit Facilities by Region



***

# **Code Graveyard**

***

In [11]:
# # Data Transformation

# passenger = data[(data['Facility'] == 'Passenger') & (data['Condition Assessment'] < 3)]
# admin = data[(data['Facility'] == 'Admin') & (data['Condition Assessment'] < 3)]

# display(passenger.head())
# display(admin.head())


In [12]:
# passenger.groupby(['Year', 'Agency Name', 'Facility Type']).size().reset_index(name='Facility Count')


In [13]:
# total_facilities = data.groupby(['Year', 'Agency Name', 'Facility']).size().reset_index(name='Total Facilities')

In [14]:
# total_facilities = data.groupby(['Year', 'Agency Name', 'Facility', 'NTD ID', 'City']).size().reset_index(name='Total Facilities')
# total_facilities

In [15]:
# data[data['Passenger/Parking Facility Flag'] == 1.0]

In [16]:
# # Count total facilities by year, agency, facility type, and NTD ID (without City grouping)
# total_facilities = data.groupby(['Year', 'Agency Name', 'Facility', 'NTD ID']).size().reset_index(name='Total Facilities')

# # Count deficient facilities (condition assessment < 3) by year, agency, facility type, and NTD ID (without City grouping)
# deficient_facilities = data[data['Condition Assessment'] < 3].groupby(['Year', 'Agency Name', 'Facility', 'NTD ID']).size().reset_index(name='Deficient Facilities')

# # Merge total and deficient facility counts
# facility_summary = total_facilities.merge(deficient_facilities, on=['Year', 'Agency Name', 'Facility', 'NTD ID'], how='left').fillna(0)

# # Calculate the percentage of deficient facilities
# facility_summary['Deficient Percentage'] = (facility_summary['Deficient Facilities'] / facility_summary['Total Facilities']) * 100

# # Merge the original data to get the 'City' column (just bringing in city data)
# facility_summary

# # Elk Grove and Folsom are alone because they have no data avaialble for later years